In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_train = pd.read_csv('/content/train_cleaned_checkpoint.csv')
df_test = pd.read_csv('/content/test_cleaned_checkpoint.csv')
df_valid = pd.read_csv('/content/valid_cleaned_checkpoint.csv')

# **Label Encoding**


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# fit_transform
y_train = label_encoder.fit_transform(df_train['label'])

# transform
y_valid = label_encoder.transform(df_valid['label'])
y_test  = label_encoder.transform(df_test['label'])

# Class Mapping
class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print("Emotion Class Mapping:", class_mapping)

Emotion Class Mapping: {'anger': 0, 'fear': 1, 'happy': 2, 'love': 3, 'sadness': 4}


# **TF-IDF Vectorization (Classical Machine Learning)**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

# fit_transform
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['clean_text_classical'])

# transform
X_valid_tfidf = tfidf_vectorizer.transform(df_valid['clean_text_classical'])
X_test_tfidf  = tfidf_vectorizer.transform(df_test['clean_text_classical'])

# Dimension
print(f"X_train_tfidf Dimension: {X_train_tfidf.shape}")

X_train_tfidf Dimension: (3521, 11480)


# **Training Model (CNB, SVM, RF)**

## **Train and Evaluation Funciton**

In [ ]:
from sklearn.metrics import classification_report, f1_score

target_names = label_encoder.classes_

def train_eval(model, model_name, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f"--- Training Model: {model_name} ---")
    model.fit(X_train, y_train)

    print("\n[Evaluation on VALID data]")
    y_pred_valid = model.predict(X_valid)
    print(classification_report(y_valid, y_pred_valid, target_names=target_names))

    print("\n[Evaluation on TEST data]")
    y_pred_test = model.predict(X_test)
    print(classification_report(y_test, y_pred_test, target_names=target_names))

    return y_pred_test

## **A. Complement Naive Bayes**

In [ ]:
from sklearn.naive_bayes import ComplementNB

cnb = ComplementNB()

y_pred_nb = train_eval(
    model=cnb,
    model_name="Complement Naive Bayes",
    X_train=X_train_tfidf, y_train=y_train,
    X_valid=X_valid_tfidf, y_valid=y_valid,
    X_test=X_test_tfidf, y_test=y_test
)

y_pred_nb = np.array(y_pred_nb)
np.save('y_pred_nb.npy', y_pred_nb)

--- Training Model: Complement Naive Bayes ---

[Evaluation on VALID data]
              precision    recall  f1-score   support

       anger       0.69      0.79      0.73       110
        fear       0.73      0.74      0.73        65
       happy       0.69      0.58      0.63       102
        love       0.57      0.75      0.65        64
     sadness       0.55      0.42      0.48        99

    accuracy                           0.65       440
   macro avg       0.64      0.66      0.64       440
weighted avg       0.64      0.65      0.64       440


[Evaluation on TEST data]
              precision    recall  f1-score   support

       anger       0.65      0.81      0.72       110
        fear       0.75      0.63      0.68        65
       happy       0.66      0.53      0.59       101
        love       0.63      0.83      0.72        64
     sadness       0.62      0.51      0.56       100

    accuracy                           0.65       440
   macro avg       0.66      

## **B. Support Vector Machine (SVM)**

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='linear', class_weight='balanced', random_state=42)

y_pred_svm = train_eval(
    model=svm,
    model_name="Support Vector Machine (Linear)",
    X_train=X_train_tfidf, y_train=y_train,
    X_valid=X_valid_tfidf, y_valid=y_valid,
    X_test=X_test_tfidf, y_test=y_test
)

y_pred_svm = np.array(y_pred_svm)
np.save('y_pred_svm.npy', y_pred_svm)

--- Training Model: Support Vector Machine (Linear) ---

[Evaluation on VALID data]
              precision    recall  f1-score   support

       anger       0.61      0.75      0.67       110
        fear       0.73      0.57      0.64        65
       happy       0.63      0.55      0.59       102
        love       0.66      0.73      0.70        64
     sadness       0.51      0.47      0.49        99

    accuracy                           0.61       440
   macro avg       0.63      0.62      0.62       440
weighted avg       0.62      0.61      0.61       440


[Evaluation on TEST data]
              precision    recall  f1-score   support

       anger       0.62      0.73      0.67       110
        fear       0.87      0.63      0.73        65
       happy       0.65      0.63      0.64       101
        love       0.78      0.77      0.77        64
     sadness       0.56      0.57      0.56       100

    accuracy                           0.66       440
   macro avg       0

## **C. Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)

y_pred_rf = train_eval(
    model=rf,
    model_name="Random Forest",
    X_train=X_train_tfidf, y_train=y_train,
    X_valid=X_valid_tfidf, y_valid=y_valid,
    X_test=X_test_tfidf, y_test=y_test
)

y_pred_rf = np.array(y_pred_rf)
np.save('y_pred_rf.npy', y_pred_rf)

--- Training Model: Random Forest ---

[Evaluation on VALID data]
              precision    recall  f1-score   support

       anger       0.53      0.74      0.62       110
        fear       0.87      0.60      0.71        65
       happy       0.64      0.52      0.57       102
        love       0.62      0.78      0.69        64
     sadness       0.52      0.41      0.46        99

    accuracy                           0.60       440
   macro avg       0.63      0.61      0.61       440
weighted avg       0.62      0.60      0.60       440


[Evaluation on TEST data]
              precision    recall  f1-score   support

       anger       0.55      0.77      0.64       110
        fear       0.89      0.65      0.75        65
       happy       0.65      0.56      0.60       101
        love       0.65      0.78      0.71        64
     sadness       0.60      0.44      0.51       100

    accuracy                           0.63       440
   macro avg       0.67      0.64     

# **Hyperparameter Tuning**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

param_dist_cnb = {
    'alpha': loguniform(1e-2, 10),
    'norm': [False, True]
}

param_dist_svm = {
    'C': loguniform(1e-2, 10)
}

param_dist_rf = {
    'n_estimators': randint(50, 200),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 11)
}

# RandomizedSearch
def run_randomized_search(model, param_dist, model_name, X_train, y_train, n_iter=10, cv=3):
    print(f"\nTuning Model: {model_name}")

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring='f1_macro',
        cv=3,
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    random_search.fit(X_train, y_train)

    print(f"Best Params: {random_search.best_params_}")
    print(f"Best Macro F1-Score (Cross-Validation): {random_search.best_score_:.4f}")

    return random_search.best_estimator_

# A. Complement Naive Bayes (n_iter=10)
best_cnb = run_randomized_search(
    ComplementNB(),
    param_dist_cnb,
    "Complement Naive Bayes",
    X_train_tfidf, y_train,
    n_iter=10, cv=3
)

# B. SVM Linear (n_iter=10)
best_svm = run_randomized_search(
    SVC(kernel='linear', class_weight='balanced', random_state=42),
    param_dist_svm,
    "SVM (Linear)",
    X_train_tfidf, y_train,
    n_iter=10, cv=3
)

# C. Random Forest (n_iter=10)
best_rf = run_randomized_search(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_dist_rf,
    "Random Forest",
    X_train_tfidf, y_train,
    n_iter=10, cv=3
)


Tuning Model: Complement Naive Bayes
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Params: {'alpha': np.float64(3.142880890840109), 'norm': True}
Best Macro F1-Score (Cross-Validation): 0.6512

Tuning Model: SVM (Linear)
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Params: {'C': np.float64(0.6251373574521749)}
Best Macro F1-Score (Cross-Validation): 0.6453

Tuning Model: Random Forest
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Params: {'max_depth': 30, 'min_samples_split': 7, 'n_estimators': 179}
Best Macro F1-Score (Cross-Validation): 0.6275


## **Train Tuned Model**

In [ ]:
print("\n[CNB-tuned Evaluation on TEST data]")
y_pred_bcnb = best_cnb.predict(X_test_tfidf)
print(classification_report(y_test, y_pred_bcnb, target_names=target_names))

print("\n[SVM-tuned Evaluation on TEST data]")
y_pred_bsvm = best_svm.predict(X_test_tfidf)
print(classification_report(y_test, y_pred_bsvm, target_names=target_names))

print("\n[RF-tuned Evaluation on TEST data]")
y_pred_brf = best_rf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred_brf, target_names=target_names))



[CNB-tuned Evaluation on TEST data]
              precision    recall  f1-score   support

       anger       0.63      0.86      0.73       110
        fear       0.91      0.60      0.72        65
       happy       0.74      0.57      0.65       101
        love       0.72      0.78      0.75        64
     sadness       0.61      0.61      0.61       100

    accuracy                           0.69       440
   macro avg       0.72      0.69      0.69       440
weighted avg       0.71      0.69      0.69       440


[SVM-tuned Evaluation on TEST data]
              precision    recall  f1-score   support

       anger       0.62      0.78      0.69       110
        fear       0.89      0.65      0.75        65
       happy       0.63      0.61      0.62       101
        love       0.77      0.73      0.75        64
     sadness       0.54      0.51      0.52       100

    accuracy                           0.65       440
   macro avg       0.69      0.66      0.67       440
wei

In [ ]:
y_pred_bcnb = np.array(y_pred_bcnb)
np.save('y_pred_bcnb.npy', y_pred_bcnb)

y_pred_bsvm = np.array(y_pred_bsvm)
np.save('y_pred_bsvm.npy', y_pred_bsvm)

y_pred_brf = np.array(y_pred_brf)
np.save('y_pred_brf.npy', y_pred_brf)